In [1]:
import os
os.chdir("/home/mdarif/MLOPS/Wine-Quality-Prediction")
print(os.getcwd())

/home/mdarif/MLOPS/Wine-Quality-Prediction


In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_file: Path
    database_table: str
    kaggle_dataset: str
    status_file: Path

In [3]:
from wine_quality_prediction.constants import *
from wine_quality_prediction.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:

    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH):

      self.config = read_yaml(config_filepath)
      self.params = read_yaml(params_filepath)
      self.schema = read_yaml(schema_filepath)
      create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self):
          config = self.config.data_ingestion
          create_directories([config.root_dir])
          return DataIngestionConfig(
                root_dir=Path(config.root_dir),
                source_file=Path(config.source_file),
                database_table=config.database_table,
                kaggle_dataset=config.kaggle_dataset,
                status_file=Path(config.status_file)
          )

In [ ]:
import os
import urllib.request
import pandas as pd
from wine_quality_prediction.database import DatabaseOperations
from wine_quality_prediction.logger import get_logger, DataIngestionError
from wine_quality_prediction.utils import get_size, save_status

logger = get_logger("data_ingestion", "data_ingestion.log")

In [ ]:
class DataIngestion:
    """Handles data downloading from URL and storing raw data into database. """

    def download_data(self):
        """Download dataset from Github raw URL."""
        try:
            if not os.path.exists(self.config.source_file):
                logger.info(f"Downloading dataset from: '{self.config.dataset_url}' ")
                urllib.request.urlretrieve(self.config.dataset_url, self.config.source_file)
                logger.info(f"Dataset downloaded successfully to: '{self.config.source_file}'")
                logger.info(f"File size: '{get_size(self.config.source_file)}'")
            else:
                logger.info(f"Dataset already exists at: '{self.config.source_file}'")

        except Exception as e:
            logger.error(
                f"Download failed - "
                f"url='{self.config.dataset_url}' "
                f"destination='{self.config.source_file}': {e}",
                exc_info=True
            )
            raise DataIngestionError("Failed to download data") from e

    def load_csv(self) -> pd.DataFrame:
        """Load raw data from CSV file."""

        try:
            file_path = self.config.source_file
            logger.info(f"Reading file: '{file_path}'")
            logger.info(f"File size: '{get_size(file_path)}'")
            df = pd.read_csv(file_path)
            logger.info(f"CSV loaded successfully with shape: {df.shape}")
            logger.info(f"Columns found: {df.columns.tolist()}")
            return df

        except Exception as e:
            logger.error(
                f"CSV load failed - "
                f"path='{self.config.source_file}': {e}",
                exc_info=True
            )
            raise DataIngestionError(f"Failed to load CSV") from e

    def store_in_database(self, df: pd.DataFrame):
        """Store raw dataframe into database table."""

        db = DatabaseOperations()
        try:
            logger.info("Creating table if not exists")
            db.create_table()
            existing = db.fetch_data()
            if len(existing) > 0:
                logger.info("Data already in database skipping insert")
                return
            logger.info("Inserting raw data into database")
            db.insert_dataframe(df)
            logger.info("Data inserted successfully.")

        except Exception as e:
            logger.error(
                f"Database insert failed - "
                f"table='{self.config.database_table}' "
                f"rows='{len(df)}': {e}",
                exc_info=True
            )
            raise DataIngestionError("Failed to store data in database") from e
        finally:
            db.close_connection()

    def run(self):
        """Run full data ingestion pipeline."""

        try:
            self.download_data()
            df = self.load_csv()
            self.store_in_database(df)
            save_status(
                self.config.status_file,
                "data_ingestion",
                True
            )
            logger.info("Data ingestion completed successfully")

        except Exception as e:
            save_status(self.config.status_file, "data_ingestion", False)
            logger.error(f"Unexpected error: {e}", exc_info=True)
            raise DataIngestionError("Data ingestion pipeline failed") from e

In [7]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.run()
except Exception:
    raise

2026-06-12 17:59:11,717 - common - INFO - common.py:31 - YAML file loaded successfully: 'config/config.yaml'
2026-06-12 17:59:11,722 - common - INFO - common.py:31 - YAML file loaded successfully: 'params.yaml'
2026-06-12 17:59:11,725 - common - INFO - common.py:31 - YAML file loaded successfully: 'schema.yaml'
2026-06-12 17:59:11,726 - common - INFO - common.py:53 - Created directory at: 'artifacts'
2026-06-12 17:59:11,726 - common - INFO - common.py:53 - Created directory at: 'artifacts/data_ingestion'
2026-06-12 17:59:11,727 - data_ingestion - INFO - 3490111120.py:14 - Downloading dataset from: 'https://raw.githubusercontent.com/arifaziz9162/datasets/refs/heads/main/wine-quality/WineQT.csv' 


2026-06-12 17:59:12,131 - data_ingestion - INFO - 3490111120.py:19 - Dataset downloaded successfully to: 'artifacts/data_ingestion/WineQT.csv'
2026-06-12 17:59:12,132 - data_ingestion - INFO - 3490111120.py:20 - File size: '~76 KB'
2026-06-12 17:59:12,133 - data_ingestion - INFO - 3490111120.py:38 - Reading file: 'artifacts/data_ingestion/WineQT.csv'
2026-06-12 17:59:12,134 - data_ingestion - INFO - 3490111120.py:39 - File size: '~76 KB'
2026-06-12 17:59:12,141 - data_ingestion - INFO - 3490111120.py:41 - CSV loaded successfully with shape: (1143, 13)
2026-06-12 17:59:12,142 - data_ingestion - INFO - 3490111120.py:42 - Columns found: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality', 'Id']
2026-06-12 17:59:12,163 - db_connection - INFO - connection.py:26 - Connected to database successfully.
2026-06-12 17:59:12,202 - db_connection - INFO - connection.py:51